# Project 9: Hospital Appointment Operations Analysis

Tasks 124-138 plus two extra analyses (age groups, weekday and month trends) and file handling.

**How to run in Google Colab:** upload `arjun.csv` using the folder icon on the left (or let Cell 2 ask you to upload it), then choose **Runtime > Run all**. Run the cells in order, because later cells use variables from earlier ones.

## Cell 1: Import libraries

In [1]:
import os
import numpy as np
import pandas as pd

## Cell 2: Find the data file

Looks for `arjun.csv` in the current folder, the Colab files area and Google Drive, and offers an upload button if it is missing.

In [2]:
CSV = "arjun.csv"

# Find the data file (works on a laptop and in Google Colab) -------------------
if os.path.exists(CSV):
    print(f"Found {CSV} in the current folder.")
elif os.path.exists("/content/arjun.csv"):
    CSV = "/content/arjun.csv"
    print(f"Found {CSV} in the Colab files area.")
elif os.path.exists("/content/drive/MyDrive/arjun.csv"):
    CSV = "/content/drive/MyDrive/arjun.csv"
    print(f"Found {CSV} in Google Drive.")
else:
    try:
        # Only works inside Google Colab: opens an upload button for the CSV.
        from google.colab import files
        print("arjun.csv not found. Please choose the file to upload...")
        uploaded = files.upload()
        CSV = list(uploaded.keys())[0]
        print(f"Using uploaded file: {CSV}")
    except ImportError:
        raise FileNotFoundError(
            "arjun.csv not found. Put it in the same folder as this script "
            "and run the script again."
        )

Found arjun.csv in the current folder.


## Cell 3: Setup (output folder, display options, helper function)

In [3]:
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)  # show every column in df.head()


def section(title):
    print("\n" + "=" * 8 + " " + title + " " + "=" * 8)

## Task 124: Load and inspect the records

Opens the CSV safely with `try / except` and checks that all 9 columns are present.

In [4]:
# 124. Load and inspect -------------------------------------------------------
section("124. Load and inspect")
# File handling: open the CSV safely and report any problem clearly.
required_columns = ["Appointment_ID", "Age", "Department", "Appointment_Date",
                    "Wait_Time_Min", "Appointment_Type", "SMS_Reminder",
                    "Attendance_Status", "Consultation_Fee"]
try:
    df = pd.read_csv(CSV)
except FileNotFoundError:
    raise SystemExit(f"Error: {CSV} could not be opened. Check the file name and folder.")
except pd.errors.EmptyDataError:
    raise SystemExit(f"Error: {CSV} is empty.")
except pd.errors.ParserError:
    raise SystemExit(f"Error: {CSV} is not a valid CSV file.")

missing_cols = [c for c in required_columns if c not in df.columns]
if len(missing_cols) > 0:
    raise SystemExit(f"Error: these columns are missing from {CSV}: {missing_cols}")
elif len(df) == 0:
    raise SystemExit(f"Error: {CSV} has headers but no rows.")
else:
    print(f"File check passed: {CSV} ({os.path.getsize(CSV):,} bytes, all 9 columns present).")

print("Shape:", df.shape)
print(df.dtypes)
print(df.head())
df["Appointment_Date"] = pd.to_datetime(df["Appointment_Date"], format="%d-%m-%Y")
print("Date range:", df["Appointment_Date"].min().date(), "to", df["Appointment_Date"].max().date())


======== 124. Load and inspect ========
File check passed: arjun.csv (59,152 bytes, all 9 columns present).
Shape: (1000, 9)
Appointment_ID        object
Age                    int64
Department            object
Appointment_Date      object
Wait_Time_Min        float64
Appointment_Type      object
SMS_Reminder          object
Attendance_Status     object
Consultation_Fee       int64
dtype: object
  Appointment_ID  Age   Department Appointment_Date  Wait_Time_Min Appointment_Type SMS_Reminder Attendance_Status  Consultation_Fee
0        AP00000   22  Dermatology       18-04-2025           29.0              New           No         Cancelled               700
1        AP00001   66   Pediatrics       13-04-2025            NaN        Follow-up          Yes          Attended               900
2        AP00002   64   Pediatrics       18-06-2025           38.0              New          Yes          Attended               500
3        AP00003   27   Cardiology       19-06-2025            NaN 

## Task 125: Identify and handle missing waiting times

In [5]:
# 125. Missing waiting times --------------------------------------------------
section("125. Missing waiting times")
print(df.isna().sum())
n_missing = df["Wait_Time_Min"].isna().sum()
# Fill with the department median (robust to outliers); keep a flag column.
df["Wait_Imputed"] = df["Wait_Time_Min"].isna()
df["Wait_Time_Min"] = df["Wait_Time_Min"].fillna(
    df.groupby("Department")["Wait_Time_Min"].transform("median")
)
print(f"Filled {n_missing} missing waiting times with department medians.")


======== 125. Missing waiting times ========
Appointment_ID        0
Age                   0
Department            0
Appointment_Date      0
Wait_Time_Min        41
Appointment_Type      0
SMS_Reminder          0
Attendance_Status     0
Consultation_Fee      0
dtype: int64
Filled 41 missing waiting times with department medians.


## Task 126: Check duplicate Appointment_ID values

In [6]:
# 126. Duplicate IDs ----------------------------------------------------------
section("126. Duplicate Appointment_ID")
dups = df["Appointment_ID"].duplicated().sum()
print("Duplicate IDs:", dups)
df = df.drop_duplicates(subset="Appointment_ID", keep="first")

# Cleaned dataset preview (after missing-value and duplicate handling)
df_cleaned = df.copy()
section("Cleaned data preview: df_cleaned.head()")
print(df_cleaned.head())
print("Missing values left in Wait_Time_Min:", df_cleaned["Wait_Time_Min"].isna().sum())


======== 126. Duplicate Appointment_ID ========
Duplicate IDs: 0

======== Cleaned data preview: df_cleaned.head() ========
  Appointment_ID  Age   Department Appointment_Date  Wait_Time_Min Appointment_Type SMS_Reminder Attendance_Status  Consultation_Fee  Wait_Imputed
0        AP00000   22  Dermatology       2025-04-18           29.0              New           No         Cancelled               700         False
1        AP00001   66   Pediatrics       2025-04-13           31.0        Follow-up          Yes          Attended               900          True
2        AP00002   64   Pediatrics       2025-06-18           38.0              New          Yes          Attended               500         False
3        AP00003   27   Cardiology       2025-06-19           23.0              New          Yes          Attended               500          True
4        AP00004   15   Cardiology       2025-04-19           22.0              New          Yes          Attended               700        

## Task 127: Total appointments and attendance-status percentages

In [7]:
# 127. Totals and attendance status % ----------------------------------------
section("127. Attendance status")
total = len(df)
status = df["Attendance_Status"].value_counts()
status_pct = (status / total * 100).round(2)
att = pd.DataFrame({"Count": status, "Percent": status_pct})
print("Total appointments:", total)
print(att)
att.to_csv(f"{OUT}/attendance_status.csv")


======== 127. Attendance status ========
Total appointments: 1000
                   Count  Percent
Attendance_Status                
Attended             803     80.3
No-show              118     11.8
Cancelled             79      7.9


## Task 128: Average waiting time by department

In [8]:
# 128. Average waiting time by department ------------------------------------
section("128. Average waiting time by department")
wait_group = df.groupby("Department")["Wait_Time_Min"]
wait_dept = pd.DataFrame({
    "Appointments": wait_group.count(),
    "Avg_Wait_Min": wait_group.mean().round(2),
})
print(wait_dept.sort_values("Avg_Wait_Min", ascending=False))
for dept, avg in wait_dept["Avg_Wait_Min"].items():
    if avg > 30:
        print(f"{dept}: {avg} min -> HIGH average wait (above 30 min)")
    elif avg > 27:
        print(f"{dept}: {avg} min -> MODERATE average wait (27-30 min)")
    else:
        print(f"{dept}: {avg} min -> LOW average wait (27 min or less)")


======== 128. Average waiting time by department ========
             Appointments  Avg_Wait_Min
Department                             
Pediatrics            187         30.32
Dermatology           204         29.92
Orthopedics           186         27.76
General               216         27.70
Cardiology            207         25.23
Cardiology: 25.23 min -> LOW average wait (27 min or less)
Dermatology: 29.92 min -> MODERATE average wait (27-30 min)
General: 27.7 min -> MODERATE average wait (27-30 min)
Orthopedics: 27.76 min -> MODERATE average wait (27-30 min)
Pediatrics: 30.32 min -> HIGH average wait (above 30 min)


## Task 129: No-show percentage by department

In [9]:
# 129. No-show % by department -----------------------------------------------
section("129. No-show percentage by department")
df["NoShow"] = (df["Attendance_Status"] == "No-show").astype(int)
ns_group = df.groupby("Department")["NoShow"]
noshow_dept = pd.DataFrame({
    "NoShows": ns_group.sum(),
    "Appointments": ns_group.count(),
    "NoShow_Pct": (ns_group.mean() * 100).round(2),
}).sort_values("NoShow_Pct", ascending=False)
print(noshow_dept)
print("Highest no-show department:", noshow_dept.index[0])
for dept, pct in noshow_dept["NoShow_Pct"].items():
    if pct >= 14:
        print(f"{dept}: {pct}% -> HIGH no-show risk (14% or more)")
    elif pct >= 10:
        print(f"{dept}: {pct}% -> MEDIUM no-show risk (10% to 14%)")
    else:
        print(f"{dept}: {pct}% -> LOW no-show risk (below 10%)")
noshow_dept.to_csv(f"{OUT}/noshow_by_department.csv")


======== 129. No-show percentage by department ========
             NoShows  Appointments  NoShow_Pct
Department                                    
Pediatrics        28           187       14.97
Dermatology       30           204       14.71
Orthopedics       27           186       14.52
Cardiology        19           207        9.18
General           14           216        6.48
Highest no-show department: Pediatrics
Pediatrics: 14.97% -> HIGH no-show risk (14% or more)
Dermatology: 14.71% -> HIGH no-show risk (14% or more)
Orthopedics: 14.52% -> HIGH no-show risk (14% or more)
Cardiology: 9.18% -> LOW no-show risk (below 10%)
General: 6.48% -> LOW no-show risk (below 10%)


## Task 130: SMS reminder Yes vs No

In [10]:
# 130. SMS reminder vs no-show ------------------------------------------------
section("130. SMS reminder Yes vs No")
ct = pd.crosstab(df["SMS_Reminder"], df["Attendance_Status"])
ct_pct = (pd.crosstab(df["SMS_Reminder"], df["Attendance_Status"], normalize="index") * 100).round(2)
print(ct)
print(ct_pct)
ct_pct.to_csv(f"{OUT}/sms_vs_attendance_pct.csv")
ns_yes = ct_pct.loc["Yes", "No-show"]
ns_no = ct_pct.loc["No", "No-show"]
if abs(ns_yes - ns_no) < 1:
    print(f"SMS Yes ({ns_yes}%) and SMS No ({ns_no}%) differ by less than 1 point: "
          "reminders make almost no difference.")
elif ns_yes < ns_no:
    print(f"SMS Yes ({ns_yes}%) has a LOWER no-show rate than SMS No ({ns_no}%): reminders appear to help.")
else:
    print(f"SMS Yes ({ns_yes}%) has a HIGHER no-show rate than SMS No ({ns_no}%): reminders show no benefit.")


======== 130. SMS reminder Yes vs No ========
Attendance_Status  Attended  Cancelled  No-show
SMS_Reminder                                   
No                      446         39       65
Yes                     357         40       53
Attendance_Status  Attended  Cancelled  No-show
SMS_Reminder                                   
No                    81.09       7.09    11.82
Yes                   79.33       8.89    11.78
SMS Yes (11.78%) and SMS No (11.82%) differ by less than 1 point: reminders make almost no difference.


## Task 131: Average consultation fee by department

In [11]:
# 131. Average consultation fee by department --------------------------------
section("131. Average consultation fee by department")
fee_dept = df.groupby("Department")["Consultation_Fee"].mean().round(2).sort_values(ascending=False)
print(fee_dept)
fee_dept.to_csv(f"{OUT}/avg_fee_by_department.csv")


======== 131. Average consultation fee by department ========
Department
General        785.19
Dermatology    775.49
Pediatrics     740.11
Cardiology     684.06
Orthopedics    674.19
Name: Consultation_Fee, dtype: float64


## Task 132: Appointment volume by date and department

In [12]:
# 132. Volume by date and department -----------------------------------------
section("132. Appointment volume by date and department")
daily = df.groupby(["Appointment_Date", "Department"]).size().unstack(fill_value=0)
daily["Total"] = daily.sum(axis=1)
print(daily.head(10))
print("Busiest dates:\n", daily["Total"].sort_values(ascending=False).head(5))
daily.to_csv(f"{OUT}/volume_by_date_department.csv")
monthly = (df.groupby([df["Appointment_Date"].dt.to_period("M"), "Department"]).size().unstack(fill_value=0))
print("Monthly volume:\n", monthly)
monthly.to_csv(f"{OUT}/volume_by_month_department.csv")


======== 132. Appointment volume by date and department ========
Department        Cardiology  Dermatology  General  Orthopedics  Pediatrics  Total
Appointment_Date                                                                  
2025-01-01                 2            2        7            0           0     11
2025-01-02                 3            0        1            0           0      4
2025-01-03                 2            0        1            0           0      3
2025-01-04                 1            1        0            0           0      2
2025-01-05                 0            1        0            0           1      2
2025-01-06                 0            0        2            0           0      2
2025-01-07                 1            0        0            0           2      3
2025-01-08                 1            1        2            0           0      4
2025-01-09                 0            0        0            0           1      1
2025-01-10           

## Task 133: Waiting time for New vs Follow-up appointments

In [13]:
# 133. Waiting time: New vs Follow-up ----------------------------------------
section("133. Waiting time by appointment type")
type_group = df.groupby("Appointment_Type")["Wait_Time_Min"]
wait_type = pd.DataFrame({
    "count": type_group.count(),
    "mean": type_group.mean().round(2),
    "median": type_group.median().round(2),
})
print(wait_type)
new_avg = wait_type.loc["New", "mean"]
fu_avg = wait_type.loc["Follow-up", "mean"]
if abs(new_avg - fu_avg) < 1:
    print("New and Follow-up waits are almost the same (difference under 1 minute).")
elif new_avg > fu_avg:
    print("New appointments wait LONGER than Follow-up appointments.")
else:
    print("Follow-up appointments wait LONGER than New appointments.")


======== 133. Waiting time by appointment type ========
                  count   mean  median
Appointment_Type                      
Follow-up           493  28.32    28.0
New                 507  27.98    27.0
New and Follow-up waits are almost the same (difference under 1 minute).


## Task 134: NumPy median, quartiles and standard deviation

In [14]:
# 134. NumPy median, quartiles, std ------------------------------------------
section("134. NumPy statistics of waiting time")
w = df["Wait_Time_Min"].to_numpy(dtype=float)
median = np.median(w)
q1, q3 = np.percentile(w, [25, 75])
std = np.std(w, ddof=1)
print(f"Median={median:.2f}  Q1={q1:.2f}  Q3={q3:.2f}  Std(sample)={std:.2f}")


======== 134. NumPy statistics of waiting time ========
Median=28.00  Q1=18.00  Q3=38.00  Std(sample)=14.47


## Task 135: Unusually long waiting times (mean + 2 standard deviations)

In [15]:
# 135. Unusually long waits: mean + 2 standard deviations (no IQR) ------------
section("135. Unusually long waits (mean + 2 standard deviations)")
mean_wait = np.mean(w)
upper = mean_wait + 2 * std
outliers = df[df["Wait_Time_Min"] > upper]
share_out = len(outliers) / len(df) * 100
print(f"Mean={mean_wait:.2f}  Std={std:.2f}  limit (mean + 2 x std)={upper:.2f}")
print(f"Outliers: {len(outliers)} ({share_out:.1f}% of appointments)")
if len(outliers) == 0:
    print("No unusually long waits found.")
elif share_out <= 5:
    print("A small share of unusually long waits: review these cases individually.")
else:
    print("A large share of unusually long waits: this suggests a systematic delay problem.")
print(outliers[["Appointment_ID", "Department", "Appointment_Type", "Wait_Time_Min"]])
outliers.to_csv(f"{OUT}/wait_outliers.csv", index=False)


======== 135. Unusually long waits (mean + 2 standard deviations) ========
Mean=28.14  Std=14.47  limit (mean + 2 x std)=57.09
Outliers: 20 (2.0% of appointments)
A small share of unusually long waits: review these cases individually.
    Appointment_ID   Department Appointment_Type  Wait_Time_Min
167        AP00167  Orthopedics        Follow-up           68.0
169        AP00169      General              New           66.0
186        AP00186  Dermatology        Follow-up           65.0
212        AP00212  Orthopedics              New           71.0
228        AP00228      General              New           61.0
254        AP00254  Dermatology              New           62.0
337        AP00337  Dermatology        Follow-up           62.0
388        AP00388  Orthopedics              New           65.0
399        AP00399      General              New           58.0
458        AP00458   Pediatrics              New           58.0
483        AP00483  Orthopedics        Follow-up           6

## Task 136: Waiting-time categories

In [16]:
# 136. Waiting-time categories ------------------------------------------------
section("136. Waiting-time bands")
bins = [0, 15, 30, 45, np.inf]
labels = ["Short (0-15)", "Moderate (16-30)", "Long (31-45)", "Very long (46+)"]
df["Wait_Band"] = pd.cut(df["Wait_Time_Min"], bins=bins, labels=labels, right=True)
bands = df["Wait_Band"].value_counts().reindex(labels)
bands_df = pd.DataFrame({"Count": bands, "Percent": (bands / len(df) * 100).round(2)})
print(bands_df)
bands_df.to_csv(f"{OUT}/waiting_time_bands.csv")
print(pd.crosstab(df["Department"], df["Wait_Band"]))


======== 136. Waiting-time bands ========
                  Count  Percent
Wait_Band                       
Short (0-15)        203     20.3
Moderate (16-30)    361     36.1
Long (31-45)        307     30.7
Very long (46+)     129     12.9
Wait_Band    Short (0-15)  Moderate (16-30)  Long (31-45)  Very long (46+)
Department                                                                
Cardiology             48                93            47               19
Dermatology            34                64            70               36
General                45                79            74               18
Orthopedics            46                69            47               24
Pediatrics             30                56            69               32


## Task 137: Estimated consultation fees for attended appointments

In [17]:
# 137. Estimated fees for attended appointments ------------------------------
section("137. Estimated fees (attended only)")
attended = df[df["Attendance_Status"] == "Attended"]
fee_group = attended.groupby("Department")["Consultation_Fee"]
fees = pd.DataFrame({
    "Attended": fee_group.count(),
    "Total_Fees": fee_group.sum(),
    "Avg_Fee": fee_group.mean().round(2),
})
print(fees)
print("Total estimated fees:", attended["Consultation_Fee"].sum())
fees.to_csv(f"{OUT}/estimated_fees_attended.csv")
lost = df[df["Attendance_Status"].isin(["No-show", "Cancelled"])]["Consultation_Fee"].sum()
print("Fee value of no-show + cancelled slots (not collected):", lost)


======== 137. Estimated fees (attended only) ========
             Attended  Total_Fees  Avg_Fee
Department                                
Cardiology        172      123800   719.77
Dermatology       163      127600   782.82
General           181      144600   798.90
Orthopedics       147       98400   669.39
Pediatrics        140      104600   747.14
Total estimated fees: 599000
Fee value of no-show + cancelled slots (not collected): 134200


## Department summary

In [18]:
# Department summary deliverable ---------------------------------------------
dept_summary = wait_dept.join(noshow_dept[["NoShows", "NoShow_Pct"]]).join(
    fee_dept.rename("Avg_Fee")).join(fees[["Total_Fees"]])
dept_summary.to_csv(f"{OUT}/department_summary.csv")
section("Department summary")
print(dept_summary)


======== Department summary ========
             Appointments  Avg_Wait_Min  NoShows  NoShow_Pct  Avg_Fee  Total_Fees
Department                                                                       
Cardiology            207         25.23       19        9.18   684.06      123800
Dermatology           204         29.92       30       14.71   775.49      127600
General               216         27.70       14        6.48   785.19      144600
Orthopedics           186         27.76       27       14.52   674.19       98400
Pediatrics            187         30.32       28       14.97   740.11      104600


## Task 138: Operational recommendations

In [19]:
# 138. Recommendations --------------------------------------------------------
section("138. Operational recommendations")
worst_ns = noshow_dept.index[0]
worst_wait = wait_dept["Avg_Wait_Min"].idxmax()
sms = ct_pct["No-show"]
recs = [
    f"Target no-show reduction in {worst_ns} ({noshow_dept.loc[worst_ns, 'NoShow_Pct']}% no-show) with "
    "confirmation calls and overbooking buffers.",
    f"SMS reminders: no-show rate is {sms.get('Yes', float('nan'))}% with SMS vs {sms.get('No', float('nan'))}% without; "
    + ("the gap is under 1 point, so test other reminder methods (calls, timing) before extending SMS."
       if abs(sms.get('Yes', 0) - sms.get('No', 0)) < 1
       else "review whether the data supports extending reminders to all patients."),
    f"Review scheduling in {worst_wait} (longest average wait {wait_dept.loc[worst_wait, 'Avg_Wait_Min']} min) "
    "and stagger arrival slots.",
    f"Investigate the {len(outliers)} unusually long waits (> {upper:.0f} min) for staffing or process causes.",
    f"Cancelled and no-show slots represent {lost:,} in unrealised fees; add a waitlist to refill them.",
]
for i, r in enumerate(recs, 1):
    print(f"{i}. {r}")
pd.Series(recs, name="Recommendation").to_csv(f"{OUT}/recommendations.csv", index=False)


======== 138. Operational recommendations ========
1. Target no-show reduction in Pediatrics (14.97% no-show) with confirmation calls and overbooking buffers.
2. SMS reminders: no-show rate is 11.78% with SMS vs 11.82% without; the gap is under 1 point, so test other reminder methods (calls, timing) before extending SMS.
3. Review scheduling in Pediatrics (longest average wait 30.32 min) and stagger arrival slots.
4. Investigate the 20 unusually long waits (> 57 min) for staffing or process causes.
5. Cancelled and no-show slots represent 134,200 in unrealised fees; add a waitlist to refill them.


# Extra analysis A: age groups

## Extra A1: Appointments per age group

In [20]:
df_extra = df_cleaned.copy()
df_extra["NoShow"] = (df_extra["Attendance_Status"] == "No-show").astype(int)
df_extra["Cancelled"] = (df_extra["Attendance_Status"] == "Cancelled").astype(int)

section("EXTRA A1. Appointments per age group")
age_bins = [-1, 12, 17, 59, np.inf]
age_labels = ["Child (0-12)", "Teen (13-17)", "Adult (18-59)", "Senior (60+)"]
df_extra["Age_Group"] = pd.cut(df_extra["Age"], bins=age_bins, labels=age_labels)
age_counts = df_extra["Age_Group"].value_counts().reindex(age_labels)
for group, n in age_counts.items():
    share = round(n / len(df_extra) * 100, 1)
    if share >= 35:
        print(f"{group}: {n} appointments ({share}%) -> LARGEST patient group")
    elif share >= 15:
        print(f"{group}: {n} appointments ({share}%) -> MEDIUM patient group")
    else:
        print(f"{group}: {n} appointments ({share}%) -> SMALL patient group")


======== EXTRA A1. Appointments per age group ========
Child (0-12): 154 appointments (15.4%) -> MEDIUM patient group
Teen (13-17): 57 appointments (5.7%) -> SMALL patient group
Adult (18-59): 499 appointments (49.9%) -> LARGEST patient group
Senior (60+): 290 appointments (29.0%) -> MEDIUM patient group


## Extra A2: Summary by age group

In [21]:
section("EXTRA A2. Summary by age group")
age_group = df_extra.groupby("Age_Group", observed=True)
age_summary = pd.DataFrame({
    "Appointments": age_group["Appointment_ID"].count(),
    "NoShow_Pct": (age_group["NoShow"].mean() * 100).round(2),
    "Cancelled_Pct": (age_group["Cancelled"].mean() * 100).round(2),
    "Avg_Wait_Min": age_group["Wait_Time_Min"].mean().round(2),
    "Avg_Fee": age_group["Consultation_Fee"].mean().round(2),
})
print(age_summary)
age_summary.to_csv(f"{OUT}/age_group_summary.csv")


======== EXTRA A2. Summary by age group ========
               Appointments  NoShow_Pct  Cancelled_Pct  Avg_Wait_Min  Avg_Fee
Age_Group                                                                    
Child (0-12)            154       12.99          12.34         28.20   812.99
Teen (13-17)             57       12.28           3.51         30.28   759.65
Adult (18-59)           499        8.02           5.61         28.39   710.82
Senior (60+)            290       17.59          10.34         27.27   724.14


## Extra A3: No-show risk by age group

In [22]:
section("EXTRA A3. No-show risk by age group")
overall_ns = round(df_extra["NoShow"].mean() * 100, 2)
print(f"Overall no-show rate: {overall_ns}%")
for group, pct in age_summary["NoShow_Pct"].items():
    if pct > overall_ns + 2:
        print(f"{group}: {pct}% -> ABOVE average no-show rate")
    elif pct < overall_ns - 2:
        print(f"{group}: {pct}% -> BELOW average no-show rate")
    else:
        print(f"{group}: {pct}% -> CLOSE to the average no-show rate")


======== EXTRA A3. No-show risk by age group ========
Overall no-show rate: 11.8%
Child (0-12): 12.99% -> CLOSE to the average no-show rate
Teen (13-17): 12.28% -> CLOSE to the average no-show rate
Adult (18-59): 8.02% -> BELOW average no-show rate
Senior (60+): 17.59% -> ABOVE average no-show rate


## Extra A4: No-show % by age group and department

In [23]:
section("EXTRA A4. No-show % by age group and department")
age_dept = (df_extra.pivot_table(index="Age_Group", columns="Department",
                                 values="NoShow", aggfunc="mean", observed=True) * 100).round(1)
print(age_dept)
age_dept.to_csv(f"{OUT}/age_group_department_noshow.csv")
flat = age_dept.to_numpy()
r, c = np.unravel_index(np.nanargmax(flat), flat.shape)
print(f"Highest cell: {age_dept.index[r]} in {age_dept.columns[c]} ({flat[r, c]}%) "
      "(small group sizes, so treat with care)")


======== EXTRA A4. No-show % by age group and department ========
Department     Cardiology  Dermatology  General  Orthopedics  Pediatrics
Age_Group                                                               
Child (0-12)         16.7         17.1     11.8         12.9         6.7
Teen (13-17)          0.0         20.0     11.1         25.0         0.0
Adult (18-59)         5.9          7.4      4.5         10.8        13.1
Senior (60+)         12.5         27.5      5.8         20.0        23.1
Highest cell: Senior (60+) in Dermatology (27.5%) (small group sizes, so treat with care)


## Extra A5: SMS reminder effect by age group

In [24]:
section("EXTRA A5. SMS reminder effect by age group (no-show %)")
age_sms = (pd.crosstab(df_extra["Age_Group"], df_extra["SMS_Reminder"],
                       values=df_extra["NoShow"], aggfunc="mean") * 100).round(2)
print(age_sms)
age_sms.to_csv(f"{OUT}/age_group_sms_effect.csv")
for group, row in age_sms.iterrows():
    if row["Yes"] < row["No"]:
        print(f"{group}: SMS Yes {row['Yes']}% < SMS No {row['No']}% -> reminders appear to help")
    elif row["Yes"] > row["No"]:
        print(f"{group}: SMS Yes {row['Yes']}% > SMS No {row['No']}% -> reminders show no benefit")
    else:
        print(f"{group}: no difference between SMS Yes and No")


======== EXTRA A5. SMS reminder effect by age group (no-show %) ========
SMS_Reminder      No    Yes
Age_Group                  
Child (0-12)   18.95   3.39
Teen (13-17)   13.64  11.43
Adult (18-59)   6.16  10.31
Senior (60+)   17.20  18.05
Child (0-12): SMS Yes 3.39% < SMS No 18.95% -> reminders appear to help
Teen (13-17): SMS Yes 11.43% < SMS No 13.64% -> reminders appear to help
Adult (18-59): SMS Yes 10.31% > SMS No 6.16% -> reminders show no benefit
Senior (60+): SMS Yes 18.05% > SMS No 17.2% -> reminders show no benefit


# Extra analysis B: weekday and monthly trends

## Extra B1: Weekday workload

In [25]:
df_extra["Weekday"] = df_extra["Appointment_Date"].dt.day_name()
df_extra["Month"] = df_extra["Appointment_Date"].dt.month_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
month_order = ["January", "February", "March", "April", "May", "June"]

section("EXTRA B1. Weekday workload")
wd_group = df_extra.groupby("Weekday")
wd = pd.DataFrame({
    "Appointments": wd_group["Appointment_ID"].count(),
    "NoShow_Pct": (wd_group["NoShow"].mean() * 100).round(2),
    "Avg_Wait_Min": wd_group["Wait_Time_Min"].mean().round(2),
}).reindex(weekday_order)
print(wd)
wd.to_csv(f"{OUT}/weekday_summary.csv")
avg_daily = wd["Appointments"].mean()
print(f"Average appointments per weekday: {avg_daily:.1f}")
for day, n in wd["Appointments"].items():
    if n > avg_daily * 1.1:
        print(f"{day}: {n} -> BUSY day (more than 10% above average)")
    elif n < avg_daily * 0.9:
        print(f"{day}: {n} -> QUIET day (more than 10% below average)")
    else:
        print(f"{day}: {n} -> NORMAL day")
print("Busiest weekday:", wd["Appointments"].idxmax(), "| Quietest weekday:", wd["Appointments"].idxmin())
print("Highest no-show weekday:", wd["NoShow_Pct"].idxmax(), f"({wd['NoShow_Pct'].max()}%)")


======== EXTRA B1. Weekday workload ========
           Appointments  NoShow_Pct  Avg_Wait_Min
Weekday                                          
Monday              136       14.71         30.78
Tuesday             163        8.59         27.18
Wednesday           152       14.47         27.63
Thursday            136       11.03         27.64
Friday              135       10.37         27.74
Saturday            143       11.89         28.09
Sunday              135       11.85         28.20
Average appointments per weekday: 142.9
Monday: 136 -> NORMAL day
Tuesday: 163 -> BUSY day (more than 10% above average)
Wednesday: 152 -> NORMAL day
Thursday: 136 -> NORMAL day
Friday: 135 -> NORMAL day
Saturday: 143 -> NORMAL day
Sunday: 135 -> NORMAL day
Busiest weekday: Tuesday | Quietest weekday: Friday
Highest no-show weekday: Monday (14.71%)


## Extra B2: Monthly trend

In [26]:
section("EXTRA B2. Monthly trend")
mt_group = df_extra.groupby("Month")
attended_extra = df_extra[df_extra["Attendance_Status"] == "Attended"]
mt = pd.DataFrame({
    "Appointments": mt_group["Appointment_ID"].count(),
    "NoShow_Pct": (mt_group["NoShow"].mean() * 100).round(2),
    "Avg_Wait_Min": mt_group["Wait_Time_Min"].mean().round(2),
    "Fees_Attended": attended_extra.groupby("Month")["Consultation_Fee"].sum(),
}).reindex(month_order)
mt["Change_vs_Prev_Month"] = mt["Appointments"].diff()
print(mt)
mt.to_csv(f"{OUT}/monthly_summary.csv")
for month, change in mt["Change_vs_Prev_Month"].items():
    if np.isnan(change):
        print(f"{month}: first month in the data (no comparison)")
    elif change > 0:
        print(f"{month}: volume UP by {int(change)} appointments")
    elif change < 0:
        print(f"{month}: volume DOWN by {int(abs(change))} appointments")
    else:
        print(f"{month}: volume unchanged")
print("Month with highest no-show rate:", mt["NoShow_Pct"].idxmax(), f"({mt['NoShow_Pct'].max()}%)")


======== EXTRA B2. Monthly trend ========
          Appointments  NoShow_Pct  Avg_Wait_Min  Fees_Attended  Change_vs_Prev_Month
Month                                                                                
January            162        7.41         27.19         102600                   NaN
February           192       20.31         28.62         106300                  30.0
March              166        7.83         30.80          96800                 -26.0
April              163        7.98         25.48         102700                  -3.0
May                183       14.21         27.95         106100                  20.0
June               134       11.19         28.83          84500                 -49.0
January: first month in the data (no comparison)
February: volume UP by 30 appointments
March: volume DOWN by 26 appointments
April: volume DOWN by 3 appointments
May: volume UP by 20 appointments
June: volume DOWN by 49 appointments
Month with highest no-show rate: Fe

## Extra B3: Busiest weekday per department

In [27]:
section("EXTRA B3. Busiest weekday per department")
dept_wd = pd.crosstab(df_extra["Department"], df_extra["Weekday"]).reindex(columns=weekday_order)
print(dept_wd)
dept_wd.to_csv(f"{OUT}/department_weekday_volume.csv")
for dept, row in dept_wd.iterrows():
    peak_day = row.idxmax()
    share = row.max() / row.sum() * 100
    if share >= 20:
        print(f"{dept}: peaks on {peak_day} ({share:.1f}% of its appointments) -> clear weekly peak")
    else:
        print(f"{dept}: peaks on {peak_day} ({share:.1f}% of its appointments) -> workload is fairly even")


======== EXTRA B3. Busiest weekday per department ========
Weekday      Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
Department                                                                 
Cardiology       23       36         40        37      16        29      26
Dermatology      31       28         29        25      34        28      29
General          28       34         38        21      31        29      35
Orthopedics      29       32         25        24      30        25      21
Pediatrics       25       33         20        29      24        32      24
Cardiology: peaks on Wednesday (19.3% of its appointments) -> workload is fairly even
Dermatology: peaks on Friday (16.7% of its appointments) -> workload is fairly even
General: peaks on Wednesday (17.6% of its appointments) -> workload is fairly even
Orthopedics: peaks on Tuesday (17.2% of its appointments) -> workload is fairly even
Pediatrics: peaks on Tuesday (17.6% of its appointments) -> workload i

# File handling

## File handling 1: Write the summary report (mode w)

In [28]:
from datetime import datetime

section("FILE HANDLING 1. Write the summary report (mode 'w')")
report_path = f"{OUT}/analysis_report.txt"
try:
    with open(report_path, "w") as f:
        f.write("HOSPITAL APPOINTMENT OPERATIONS - SUMMARY REPORT\n")
        f.write("=" * 50 + "\n")
        f.write(f"Source file: {CSV}\n")
        f.write(f"Total appointments: {total}\n\n")
        f.write("Attendance status:\n")
        for status_name, row in att.iterrows():
            f.write(f"  {status_name}: {int(row['Count'])} ({row['Percent']}%)\n")
        f.write("\nDepartment highlights:\n")
        f.write(f"  Highest no-show rate: {worst_ns} ({noshow_dept.loc[worst_ns, 'NoShow_Pct']}%)\n")
        f.write(f"  Longest average wait: {worst_wait} ({wait_dept.loc[worst_wait, 'Avg_Wait_Min']} min)\n")
        f.write("\nWaiting time statistics (minutes):\n")
        f.write(f"  Median={median:.2f}  Q1={q1:.2f}  Q3={q3:.2f}  Std={std:.2f}\n")
        f.write(f"  Unusually long waits (> {upper:.2f}): {len(outliers)}\n")
        f.write("\nFees:\n")
        f.write(f"  Estimated fees (attended): {attended['Consultation_Fee'].sum():,}\n")
        f.write(f"  Not collected (no-show + cancelled): {lost:,}\n")
        f.write("\nRecommendations:\n")
        for i, r in enumerate(recs, 1):
            f.write(f"  {i}. {r}\n")
    print(f"Report written to {report_path}")
except OSError as error:
    print("Could not write the report:", error)


======== FILE HANDLING 1. Write the summary report (mode 'w') ========
Report written to outputs/analysis_report.txt


## File handling 2: Append to the run log (mode a)

In [29]:
section("FILE HANDLING 2. Append to the run log (mode 'a')")
log_path = f"{OUT}/run_log.txt"
try:
    with open(log_path, "a") as f:
        stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"{stamp} | file={CSV} | rows={total} | no-show%="
                f"{round(df['NoShow'].mean() * 100, 2)} | long waits={len(outliers)}\n")
    print(f"Run recorded in {log_path} (each run adds one new line).")
except OSError as error:
    print("Could not write the log:", error)


======== FILE HANDLING 2. Append to the run log (mode 'a') ========
Run recorded in outputs/run_log.txt (each run adds one new line).


## File handling 3: Read the report back (mode r)

In [30]:
section("FILE HANDLING 3. Read the report back (mode 'r')")
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        lines = f.readlines()
    print(f"{report_path} has {len(lines)} lines. First 8 lines:")
    for line in lines[:8]:
        print("  " + line.rstrip())
else:
    print("Report file not found.")


======== FILE HANDLING 3. Read the report back (mode 'r') ========
outputs/analysis_report.txt has 28 lines. First 8 lines:
  HOSPITAL APPOINTMENT OPERATIONS - SUMMARY REPORT
  Source file: arjun.csv
  Total appointments: 1000
  
  Attendance status:
    Attended: 803 (80.3%)
    No-show: 118 (11.8%)


## File handling 4: Read the run log

In [31]:
section("FILE HANDLING 4. Read the run log")
if os.path.exists(log_path):
    with open(log_path, "r") as f:
        log_lines = f.read().splitlines()
    print(f"The program has been run {len(log_lines)} time(s). Latest entry:")
    print("  " + log_lines[-1])
else:
    print("Run log not found.")


======== FILE HANDLING 4. Read the run log ========
The program has been run 1 time(s). Latest entry:
  2026-11-21 03:19:18 | file=arjun.csv | rows=1000 | no-show%=11.8 | long waits=20


## File handling 5: List the files in the outputs folder

In [32]:
section("FILE HANDLING 5. Files in the outputs folder")
for name in sorted(os.listdir(OUT)):
    size = os.path.getsize(os.path.join(OUT, name))
    if name.endswith(".csv"):
        kind = "table"
    elif name.endswith(".txt"):
        kind = "text"
    else:
        kind = "other"
    print(f"  {name:<40} {size:>7,} bytes  [{kind}]")


======== FILE HANDLING 5. Files in the outputs folder ========
  age_group_department_noshow.csv              210 bytes  [table]
  age_group_sms_effect.csv                     115 bytes  [table]
  age_group_summary.csv                        233 bytes  [table]
  analysis_report.txt                        1,144 bytes  [text]
  attendance_status.csv                         84 bytes  [table]
  avg_fee_by_department.csv                    117 bytes  [table]
  department_summary.csv                       290 bytes  [table]
  department_weekday_volume.csv                227 bytes  [table]
  estimated_fees_attended.csv                  181 bytes  [table]
  monthly_summary.csv                          276 bytes  [table]
  noshow_by_department.csv                     160 bytes  [table]
  recommendations.csv                          557 bytes  [table]
  run_log.txt                                   81 bytes  [text]
  sms_vs_attendance_pct.csv                     81 bytes  [table]
  volume_by_da